# ATC Multi-Agent GRPO Training — Jupyter Server

**Run order:** top to bottom. Edit **§0 Config** first.

**Default pipeline:** (1) **SFT** on solver gold JSON (`training/train_sft.py`) so the model learns valid `arrival_slots` / `departure_slots` / supervisor JSON before RL. (2) **GRPO** with **live grounded** training: `CurriculumManager` samples continuous `d`, softmax templates, structural variation, then **materializes** a bounded row buffer for TRL (same idea as `Dataset.from_list`, not a separate “lite” code path). Each episode emits a **full 6-role pack** — AMAN, DMAN, GENERATOR, SUPERVISOR, and **two** ADAPT rows — so grounded training includes the whole multi-agent stack (`training/live_curriculum.py`, `training/roster_integrity.py`). Matches `python training/train_grpo.py --grounded_curriculum` **without** `--static_grounded_dataset`, plus `--adapter_in` to SFT output when `RUN_SFT_PHASE` is enabled.

**Abridged path (opt-in only):** set `STATIC_GROUNDED_DATASET = True` for a pre-built static HF list (`--static_grounded_dataset`), or `RELAX_ROSTER = True` to disable strict roster asserts — for tiny smoke / debugging only, not the default.

## 0. Config — edit before running

In [ ]:
from pathlib import Path
import os


def _find_repo_root() -> Path:
    """CWD may be repo root or `training/`; locate directory containing `training/train_grpo.py`."""
    p = Path(".").resolve()
    for _ in range(6):
        if (p / "training" / "train_grpo.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path(".").resolve()


# ── Paths ──────────────────────────────────────────────────────────────────
REPO_DIR = _find_repo_root()
TRAIN_GRPO_SCRIPT = REPO_DIR / "training" / "train_grpo.py"
TRAIN_SFT_SCRIPT  = REPO_DIR / "training" / "train_sft.py"
OUTPUT_DIR = Path("/tmp/atc/outputs")     # change to a persistent path if needed
LOGS_DIR   = Path("/tmp/atc/logs")

# ── Model & training ───────────────────────────────────────────────────────
# Smoke: tiny model for a quick subprocess check. Main run: 3B default (good on 16–24GB); set 7B if you have VRAM.
SMOKE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
# Live grounded training scales max_steps from EPISODES (see training/live_curriculum.live_max_steps). Use ≥80.
EPISODES      = 80
N_GENERATIONS = 4
SEED          = 42
SKIP_SMOKE    = False       # False = run 1-episode smoke before main training

# Grounded + continuous curriculum + full multi-agent roster (defaults = real stack, not abridged)
USE_GROUNDED_CURRICULUM = True    # --grounded_curriculum: gc_* + continuous d + GENERATOR + SUP + 2×ADAPT / ep (see live_curriculum.py)
STATIC_GROUNDED_DATASET = False   # True = --static_grounded_dataset (fixed HF list; opt-in ablation). False = live materialized buffer (default).
RELAX_ROSTER = False              # True = set ATC_RELAX_ROSTER=1 (skip strict 6-pack asserts). False = strict roster (default).
CURRICULUM_STATE = None           # optional warm-start continuous_curriculum_state.json (passed as --curriculum_state when set)

# SFT → GRPO: JSON imitation on solver labels (same base model as GRPO)
RUN_SFT_PHASE     = True          # False = skip SFT; GRPO starts from fresh LoRA
SFT_OUTPUT_DIR    = OUTPUT_DIR / "atc-sft-json"
SFT_N_EPISODES    = 120           # synthetic episodes → ~3 SFT rows each
SFT_MAX_STEPS     = 400
SFT_BATCH         = 2
SFT_GRAD_ACCUM    = 4

# Throughput: reduce BATCH_SIZE on T4 (e.g. 2–4); 8 is fine on A100/L40
BATCH_SIZE      = 8
GRAD_ACCUM      = 2
MAX_NEW_TOKENS  = 384
TEMPERATURE     = 0.7
LOGGING_STEPS   = 1         # print live logs every optimizer step
EVAL_EPISODES   = 3
STRICT_GATES    = True      # fail early when parse/variance/quality gates fail

# ── W&B (optional) ─────────────────────────────────────────────────────────
# Leave blank to run offline. Or set key directly here.
WANDB_KEY    = ""         # e.g. "abcdef123..."  (overrides .env)
WANDB_PROJECT = "atc-multiagent-grpo"

# ── Derived ────────────────────────────────────────────────────────────────
SMOKE_OUTPUT_DIR = OUTPUT_DIR / "atc-smoke"
TRAIN_OUTPUT_DIR = OUTPUT_DIR / "atc-multiagent"
PLOTS_DIR        = OUTPUT_DIR / "plots"

for d in (OUTPUT_DIR, LOGS_DIR, SMOKE_OUTPUT_DIR, TRAIN_OUTPUT_DIR, PLOTS_DIR, SFT_OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not TRAIN_SFT_SCRIPT.is_file():
    raise FileNotFoundError(f"train_sft.py not found at {TRAIN_SFT_SCRIPT}")
if not TRAIN_GRPO_SCRIPT.is_file():
    raise FileNotFoundError(f"train_grpo.py not found at {TRAIN_GRPO_SCRIPT} (cwd={Path.cwd()})")


def _train_grpo_cmd(*cli_args):
    """Invoke training/train_grpo.py with the same interpreter (subprocess helpers)."""
    import sys as _sys

    return [_sys.executable, str(TRAIN_GRPO_SCRIPT), *cli_args]


def _train_sft_cmd(*cli_args):
    import sys as _sys

    return [_sys.executable, str(TRAIN_SFT_SCRIPT), *cli_args]


print(f"REPO_DIR          : {REPO_DIR}")
print(f"TRAIN_GRPO_SCRIPT : {TRAIN_GRPO_SCRIPT}")
print(f"OUTPUT_DIR        : {OUTPUT_DIR}")
print(f"PLOTS_DIR         : {PLOTS_DIR}")
print(f"USE_GROUNDED_CURRICULUM : {USE_GROUNDED_CURRICULUM}")
print(f"STATIC_GROUNDED_DATASET : {STATIC_GROUNDED_DATASET}  (False => live materialized + 6-role packs + CurriculumManager)")
print(f"RELAX_ROSTER            : {RELAX_ROSTER}  (False => strict roster asserts in train_grpo)")
print(f"RUN_SFT_PHASE     : {RUN_SFT_PHASE}  → {SFT_OUTPUT_DIR}")

## 1. Environment variables

Run **§0 Config** first so `STATIC_GROUNDED_DATASET` and `RELAX_ROSTER` exist; this cell syncs them into `os.environ` for the whole kernel.

In [ ]:
import os, socket, time

os.environ["MASTER_ADDR"]                = "127.0.0.1"
os.environ["MASTER_PORT"]                = str(30000 + (os.getpid() % 2000))
os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["PIP_NO_CACHE_DIR"]           = "1"
os.environ["NCCL_DEBUG"]                 = "INFO"
os.environ["NCCL_IB_DISABLE"]            = "1"
os.environ["NCCL_P2P_DISABLE"]           = "0"
os.environ["OMP_NUM_THREADS"]            = "4"
os.environ["MKL_NUM_THREADS"]            = "4"
os.environ["PYTHONUNBUFFERED"]           = "1"
# CRITICAL: disable torch.compile to avoid Dynamo errors with GRPO
os.environ["TORCH_COMPILE_DISABLE"]      = "1"
# OpenEnv hackathon winners (kube-sre-gym): allocator + TRL rollout noise
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
# Unsloth HF probe (120s) before model load — not your model; error text is a generic template.
# train_grpo.py also sets this if unset. Set to "0" only if you want Unsloth telemetry + HF check.
os.environ.setdefault("UNSLOTH_DISABLE_STATISTICS", "1")
# Match §0 Config (run Config before this cell): live materialized vs static list; strict vs relaxed roster.
if STATIC_GROUNDED_DATASET:
    os.environ["ATC_STATIC_GROUNDED_DATASET"] = "1"
else:
    os.environ.pop("ATC_STATIC_GROUNDED_DATASET", None)
if RELAX_ROSTER:
    os.environ["ATC_RELAX_ROSTER"] = "1"
else:
    os.environ.pop("ATC_RELAX_ROSTER", None)
print(f"Synced ATC_* : STATIC_GROUNDED={STATIC_GROUNDED_DATASET}  RELAX_ROSTER={RELAX_ROSTER}")
os.environ["WORLD_SIZE"]                 = "1"
os.environ["RANK"]                       = "0"
os.environ["LOCAL_RANK"]                 = "0"
os.environ["LOCAL_WORLD_SIZE"]           = "1"
os.environ["PYTHONPATH"]                 = str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", "")

print(f"Hostname       : {socket.gethostname()}")
print(f"Start time     : {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"MASTER_PORT    : {os.environ['MASTER_PORT']}")
print(f"TORCH_COMPILE_DISABLE=1  (Dynamo disabled)")

## 2. GPU check

In [ ]:
import subprocess, sys

subprocess.run(["nvidia-smi"], check=False)
print(f"Python: {sys.version}")

## 3. Load W&B key from .env (if present)

In [ ]:
import re

def _load_dotenv(path):
    """Minimal .env loader — no external deps needed."""
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        m = re.match(r'^(?:export\s+)?([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.*)$', line)
        if m:
            k, v = m.group(1), m.group(2).strip().strip('"\'')
            if k not in os.environ:   # don't overwrite already-set vars
                os.environ[k] = v

for env_file in (REPO_DIR / ".env", REPO_DIR / ".env.wandb"):
    _load_dotenv(env_file)
    if env_file.exists():
        print(f"Loaded: {env_file}")

# Explicit key in config cell takes priority
if WANDB_KEY.strip():
    os.environ["WANDB_API_KEY"] = WANDB_KEY.strip()

# Normalize key: strip assignment prefix + whitespace (common .env mistake)
raw_key = os.environ.get("WANDB_API_KEY", "")
raw_key = raw_key.lstrip("\r").split("=")[-1].strip()
if raw_key:
    os.environ["WANDB_API_KEY"]  = raw_key
    os.environ["WANDB_MODE"]     = "online"
    os.environ["WANDB_PROJECT"]  = WANDB_PROJECT
    print(f"W&B key found (len={len(raw_key)}) → mode=online")
else:
    os.environ.pop("WANDB_API_KEY", None)
    os.environ["WANDB_MODE"] = "offline"
    print("W&B offline (no key)")

## 4. Clear stale Unsloth pycache

In [ ]:
import shutil

removed = 0
for p in REPO_DIR.rglob("__pycache__"):
    try:
        shutil.rmtree(p)
        removed += 1
    except Exception:
        pass
print(f"Cleared {removed} __pycache__ dirs")

## 5. Install constraint-compatible packages

This cell follows package metadata constraints for the **locked preinstalled runtime**.
It avoids pins that conflict with `transformers==5.5.4` (notably `huggingface-hub==0.36.2`).

In [ ]:
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def pip(*args):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("+", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=False)
    if result.returncode != 0:
        raise RuntimeError(f"pip failed: {' '.join(args[:4])}")


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


print("Detected before install:")
for pkg in [
    "torch", "transformers", "accelerate", "peft", "bitsandbytes", "xformers",
    "trl", "huggingface-hub", "datasets", "multiprocess", "tokenizers",
]:
    print(f"  {pkg:<16} {v(pkg)}")

tf_ver = Version(v("transformers", "0"))

print("\nInstalling vllm for TRL GRPO import-time dependency...")
pip("install", "--upgrade", "--no-input", "vllm==0.19.1")

# Core fix for your traceback: transformers 5.5.4 requires huggingface-hub >= 1.5.0,<2.0
print("\nInstalling runtime-compatible utility deps...")
pip(
    "install", "--upgrade", "--no-input",
    "huggingface-hub>=1.5.0,<2.0",
    "hf_transfer==0.1.9",
    "datasets==4.8.4",
    "multiprocess==0.70.19",
    "xxhash==3.6.0",
    "tyro==0.9.17",
    "wandb==0.19.11",
)

# Keep TRL explicit for notebook behavior consistency.
pip("install", "--upgrade", "--no-input", "trl==0.16.0")

# Unsloth 2026.4.8 metadata caps transformers at <= 5.5.0.
# With the locked transformers==5.5.4 runtime, the published wheel is not metadata-compatible.
print(
    "\nUnsloth install is blocked by locked transformers="
    + str(tf_ver)
    + "; published unsloth 2026.4.8 metadata requires <= 5.5.0."
)

print("\nDone.")

## 6. Verify installations

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version


def v(pkg, default="(not installed)"):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return default


# Force-reimport after pip install in same process
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ("unsloth", "trl", "peft", "accelerate", "bitsandbytes", "transformers", "huggingface_hub", "vllm")):
        del sys.modules[mod]

import torch
import transformers
import huggingface_hub
import vllm
import trl

print(f"PyTorch        : {torch.__version__}")
print(f"Transformers   : {transformers.__version__}")
print(f"vLLM           : {vllm.__version__}")
print(f"TRL            : {trl.__version__}")
print(f"HF Hub         : {huggingface_hub.__version__}")
print(f"Datasets       : {v('datasets')}")
print(f"Multiprocess   : {v('multiprocess')}")
print(f"CUDA           : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

tf_ver = Version(transformers.__version__)
try:
    import unsloth
    from unsloth import FastLanguageModel
    print(f"Unsloth        : {unsloth.__version__}")
except Exception as exc:
    print(
        "Unsloth import failed under the locked runtime (transformers="
        f"{tf_ver}). Published unsloth 2026.4.8 metadata requires transformers<=5.5.0. "
        f"Error: {exc}"
    )

from trl import GRPOConfig, GRPOTrainer
print("All core imports OK")

## 7. W&B login

In [ ]:
if os.environ.get("WANDB_MODE") == "online":
    try:
        import wandb
        wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
        print(f"W&B logged in  project={WANDB_PROJECT}")
    except Exception as exc:
        print(f"W&B login failed ({exc}) → switching to offline")
        os.environ["WANDB_MODE"] = "offline"
else:
    print("W&B offline")

## 8. Smoke gun — 1-episode sanity run

Fast check that subprocess + model load + GRPO loop work. Set `SKIP_SMOKE = True` in Config to skip.

With grounded on: same flags as the main run — `--grounded_curriculum`, optional `--static_grounded_dataset` only if `STATIC_GROUNDED_DATASET` is True, and the same `ATC_*` env from §1 (live materialized + full 6-role roster by default).

In [ ]:
import subprocess

if not SKIP_SMOKE:
    print(f"===== SMOKE GUN START (model={SMOKE_MODEL}, episodes=1) =====")
    smoke_args = [
        "--model",        SMOKE_MODEL,
        "--output_dir",   str(SMOKE_OUTPUT_DIR),
        "--episodes",     "1",
        "--n_generations","2",
        "--seed",         str(SEED),
        "--no_eval",
    ]
    if USE_GROUNDED_CURRICULUM:
        smoke_args.append("--grounded_curriculum")
        if STATIC_GROUNDED_DATASET:
            smoke_args.append("--static_grounded_dataset")
        if CURRICULUM_STATE:
            smoke_args.extend(["--curriculum_state", str(CURRICULUM_STATE)])
    result = subprocess.run(
        _train_grpo_cmd(*smoke_args),
        env=os.environ,
        cwd=str(REPO_DIR),
    )
    if result.returncode != 0:
        raise RuntimeError(f"Smoke gun FAILED (exit {result.returncode}) — fix before full run")
    print("===== SMOKE GUN COMPLETE =====")
else:
    print("Smoke gun skipped (SKIP_SMOKE=True)")

## 9. Main training run

In [ ]:
import subprocess
import time

print(f"===== START TRAINING =====")
print(f"Model         : {TRAIN_MODEL}")
print(f"Episodes      : {EPISODES}")
print(f"Generations   : {N_GENERATIONS}")
print(f"Batch/Accum   : {BATCH_SIZE}/{GRAD_ACCUM}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print(f"Temperature   : {TEMPERATURE}")
print(f"Logging steps : {LOGGING_STEPS}")
print(f"Eval episodes : {EVAL_EPISODES}")
print(f"Strict gates  : {STRICT_GATES}")
print(f"Grounded      : {USE_GROUNDED_CURRICULUM}")
print(f"Static HF list: {STATIC_GROUNDED_DATASET}  (False = live materialized + 6-role roster)")
print(f"Relax roster  : {RELAX_ROSTER}  (False = strict pack asserts)")
print(f"Output        : {TRAIN_OUTPUT_DIR}")
print()

if RUN_SFT_PHASE:
    print("===== SFT (JSON formatting) =====")
    sft_args = [
        "--model", TRAIN_MODEL,
        "--output_dir", str(SFT_OUTPUT_DIR),
        "--n_episodes", str(SFT_N_EPISODES),
        "--max_steps", str(SFT_MAX_STEPS),
        "--batch_size", str(SFT_BATCH),
        "--grad_accum", str(SFT_GRAD_ACCUM),
        "--seed", str(SEED),
    ]
    if CURRICULUM_STATE:
        sft_args.extend(["--curriculum_state", str(CURRICULUM_STATE)])
    sft_res = subprocess.run(
        _train_sft_cmd(*sft_args),
        env=os.environ,
        cwd=str(REPO_DIR),
    )
    if sft_res.returncode != 0:
        raise RuntimeError(f"SFT FAILED (exit {sft_res.returncode})")
    print("===== SFT COMPLETE =====")

train_env = dict(os.environ)
train_env["ATC_STRICT_GATES"] = "1" if STRICT_GATES else "0"
if STATIC_GROUNDED_DATASET:
    train_env["ATC_STATIC_GROUNDED_DATASET"] = "1"
else:
    train_env.pop("ATC_STATIC_GROUNDED_DATASET", None)
if RELAX_ROSTER:
    train_env["ATC_RELAX_ROSTER"] = "1"
else:
    train_env.pop("ATC_RELAX_ROSTER", None)

train_args = [
    "--model",           TRAIN_MODEL,
    "--output_dir",      str(TRAIN_OUTPUT_DIR),
    "--episodes",        str(EPISODES),
    "--n_generations",   str(N_GENERATIONS),
    "--batch_size",      str(BATCH_SIZE),
    "--grad_accum",      str(GRAD_ACCUM),
    "--max_new_tokens",  str(MAX_NEW_TOKENS),
    "--temperature",     str(TEMPERATURE),
    "--logging_steps",   str(LOGGING_STEPS),
    "--eval_episodes",   str(EVAL_EPISODES),
    "--seed",            str(SEED),
]
if USE_GROUNDED_CURRICULUM:
    train_args.append("--grounded_curriculum")
    if STATIC_GROUNDED_DATASET:
        train_args.append("--static_grounded_dataset")
    if CURRICULUM_STATE:
        train_args.extend(["--curriculum_state", str(CURRICULUM_STATE)])
if RUN_SFT_PHASE and (SFT_OUTPUT_DIR / "adapter_config.json").is_file():
    train_args.extend(["--adapter_in", str(SFT_OUTPUT_DIR)])

t0 = time.monotonic()
result = subprocess.run(
    _train_grpo_cmd(*train_args),
    env=train_env,
    cwd=str(REPO_DIR),
)
elapsed = time.monotonic() - t0

if result.returncode != 0:
    raise RuntimeError(f"Training FAILED (exit {result.returncode})")
print(f"===== TRAINING COMPLETE ({elapsed/60:.1f} min) =====")

## 10. Generate plots

In [ ]:
import json
import sys

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import matplotlib
matplotlib.use("Agg")
from training.plot_rewards import plot_all_training_artifacts

# ── Grounded / live curriculum artifacts (paths only; figures from plot_all) ─
grounded_path = TRAIN_OUTPUT_DIR / "grounded_dataset_summary.json"
if grounded_path.exists():
    gsum = json.loads(grounded_path.read_text())
    print(f"grounded_dataset_summary.json: {gsum}")
for rel in (
    "continuous_curriculum_state.json",
    "continuous_curriculum_log.jsonl",
    "curriculum_effective_distribution.jsonl",
):
    p = TRAIN_OUTPUT_DIR / rel
    if p.exists():
        print(f"Curriculum artifact: {p}  (size={p.stat().st_size} bytes)")

generated = plot_all_training_artifacts(TRAIN_OUTPUT_DIR, PLOTS_DIR, show=False)
if not (TRAIN_OUTPUT_DIR / "reward_curves.json").exists():
    print(f"[WARN] {TRAIN_OUTPUT_DIR / 'reward_curves.json'} not found — curves skipped")

print(f"\nGenerated {len(generated)} plot file(s) → {PLOTS_DIR}")

## 11. Display plots

In [ ]:
from IPython.display import Image, display

for png in sorted(PLOTS_DIR.glob("*.png")):
    print(f"── {png.name} ──")
    display(Image(filename=str(png), width=900))

## 12. Output file summary

In [ ]:
print(f"Output directory: {TRAIN_OUTPUT_DIR}")
print()
for f in sorted(TRAIN_OUTPUT_DIR.rglob("*")):
    if f.is_file():
        size = f.stat().st_size
        unit = "KB" if size < 1_000_000 else "MB"
        val  = size / 1_000 if size < 1_000_000 else size / 1_000_000
        print(f"  {f.relative_to(TRAIN_OUTPUT_DIR):<50}  {val:6.1f} {unit}")

print()
print(f"Plots directory: {PLOTS_DIR}")
for f in sorted(PLOTS_DIR.glob("*.png")):
    print(f"  {f.name}")